In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---- Repo and environment ----
REPO = Path('/users/1/andra104/Documents/Research/Transient_Metrics/SLSNe_Metric')
sys.path.insert(0, str(REPO / 'py_files'))

from slsn_metrics.paths import get_output_dir
from slsn_metrics.runners import generate_SLSN_PopSlicer

# ---- Apply style file ----
plt.style.use(str(REPO / 'slsne_paper.mplstyle'))
print('Style file loaded:', REPO / 'slsne_paper.mplstyle')

# ---- Canonical dicts ----
MODELS = [
    'naive_physical',
    'o_dependent_physical',
    'fe_dependent_physical',
]
MODEL_LABELS = {
    'naive_physical':        'Naive SFR',
    'o_dependent_physical':  'O-dependent',
    'fe_dependent_physical': 'Fe-dependent',
}
MODEL_COLORS = {
    'naive_physical':        '#2196F3',
    'o_dependent_physical':  '#FF9800',
    'fe_dependent_physical': '#4CAF50',
}
CADENCES = ['baseline_v5.1.1_10yrs', 'baseline_v5.3.0_10yrs']
CADENCE_LABELS = {
    'baseline_v5.1.1_10yrs': 'Baseline v5.1.1',
    'baseline_v5.3.0_10yrs': 'Baseline v5.3.0',
}
SURVEY_YEARS = 10

# ---- find_latest_npy ----
def find_latest_npy(model, cadence, metric):
    """Find most recent datestamped z0.1-5.0 npy file."""
    model_dir = get_output_dir('SLSNe', subdir=model)
    metric_full = {
        'detect':       'SLSN_Detect_Physical_Metric',
        'characterize': 'SLSN_Characterize_Physical_Metric',
        'villar':       'SLSN_Villar_Physical_Metric',
        'elasticc':     'SLSN_ELAsTiCC_Metric',
        'spectrigger':  'SLSN_SpecTrigger_Physical_Metric',
    }.get(metric, metric)
    pattern = f'metric_values_{metric_full}_{model}_{cadence}_z0.1-5.0_*.npy'
    matches = sorted(model_dir.glob(pattern))
    return matches[-1] if matches else None

# ---- Load population slicers ----
import joblib
SHARED_DIR = get_output_dir('SLSNe', subdir='shared')
templates   = joblib.load(str(SHARED_DIR / 'physical_templates.pkl'))

pop_slicers = {}
for model in MODELS:
    pkl = SHARED_DIR / f'population_{model}.pkl'
    sp  = joblib.load(str(pkl))
    # Wrap as a simple namespace so slice_points works
    class _Slicer:
        def __init__(self, d): self.slice_points = d
    pop_slicers[model] = _Slicer(sp)
    print(f'Loaded {model}: {len(sp["z"]):,} events')

print('\nSetup complete.')

ImportError: cannot import name 'generate_SLSN_PopSlicer' from 'slsn_metrics.runners' (/users/1/andra104/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/slsn_metrics/runners.py)

In [ ]:
# Money plot — style file test
# Checks: serif font, inward ticks, minor ticks, grid, figure size, 300dpi save

MONEY_METRIC = 'elasticc'
MONEY_BINS   = np.linspace(0.1, 5.0, 50)

fig, axes = plt.subplots(1, len(CADENCES), sharey=True)
if len(CADENCES) == 1:
    axes = [axes]

for ax, cadence in zip(axes, CADENCES):
    detected_z = {}

    for model in MODELS:
        sp    = pop_slicers[model].slice_points
        z_all = np.asarray(sp['z'])
        npy_path = find_latest_npy(model, cadence, 'elasticc')
        if npy_path is None:
            print(f'[skip] {model}/{cadence} — no npy found')
            continue
        arr    = np.load(str(npy_path))
        mask_z = (z_all >= 0.1) & (z_all <= 5.0)
        if len(arr) != mask_z.sum():
            print(f'[skip] {model}/{cadence} — size mismatch')
            continue
        detected_z[model] = z_all[mask_z][arr == 1]

    for model in MODELS:
        if model not in detected_z:
            continue
        z_det = detected_z[model]
        color = MODEL_COLORS[model]
        label = MODEL_LABELS[model]
        ax.hist(z_det, bins=MONEY_BINS,
                histtype='stepfilled', alpha=0.25, color=color)
        ax.hist(z_det, bins=MONEY_BINS,
                histtype='step', lw=2.0, color=color,
                label=f'{label}  (N={len(z_det):,})')
        z_sorted = np.sort(z_det)
        z95 = z_sorted[int(0.95 * len(z_sorted))]
        ax.axvline(z95, color=color, ls='--', lw=1.5,
                   label=f'z(95%) = {z95:.2f}')

    ax.set_yscale('log')
    ax.set_xlabel('Redshift z')
    ax.set_ylabel('Detected SLSNe-I (count, log scale)')
    ax.set_title(f'{CADENCE_LABELS[cadence]} | ELAsTiCC')
    ax.set_xlim(0.1, 5.0)
    ax.set_xticks(np.arange(0.2, 5.1, 0.2))
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='upper right')

fig.suptitle('Detected SLSNe-I Redshift Distribution — Style File Test')

out = get_output_dir('SLSNe') / 'money_plot_style_test.png'
fig.savefig(str(out))
print(f'Saved: {out.name}')

import subprocess
r = subprocess.run(['identify', '-format', '%wx%h %x x %y dpi', str(out)],
                   capture_output=True, text=True)
print(f'Dimensions and DPI: {r.stdout}')
plt.show()